# Flow Matching

A self-contained refresher: how flow matching trains continuous-time generative
models by regressing a velocity field, and why it has largely displaced the
score-matching/SDE machinery of classic diffusion.

**Domain:** Architectures  ·  **recommended addition**  ·  **runnable:** yes

## 1. What & Why

**What it is.** Flow Matching (FM) is a way to train a *continuous normalizing flow* —
an ODE $\frac{dx}{dt} = v_\theta(x, t)$ that transports a simple base distribution
$p_0$ (usually standard Gaussian noise) into a complex data distribution $p_1$ — by
**directly regressing the velocity field** $v_\theta$ with a plain mean-squared-error
loss. No likelihoods, no SDE, no Jacobian determinants.

**The problem it solves.** Two earlier families each had a sharp edge:

- **Continuous normalizing flows (CNFs)** are elegant (an ODE you can integrate both
  ways) but were trained by maximum likelihood, which needs expensive ODE solves and
  divergence estimates *inside the training loop*. Slow and finicky.
- **Diffusion / score-based models** are trained with a cheap simulation-free loss, but
  the derivation goes through stochastic differential equations, score functions, and
  noise schedules — a lot of conceptual baggage, and sampling needs many steps.

Flow Matching keeps the cheap, **simulation-free** training of diffusion (you never
integrate the ODE during training) while producing a clean deterministic ODE for
sampling, and it generalizes the noise→data path so you can pick *straight-line* paths
that are fast to integrate (the "rectified flow" idea). In practice it trains stably,
samples in few steps, and underpins recent systems (Stable Diffusion 3, Meta's
Voicebox/Audiobox, many "rectified flow" image/video models).

**When to reach for it.** You want a generative model over continuous data
(images, audio, latents, molecular coordinates, robot trajectories), you want
deterministic few-step sampling, and you'd rather regress a vector field than reason
about score functions and SDE noise schedules.

**When not to.** Discrete data (text, tokens) — FM is native to continuous spaces;
discrete variants exist but autoregressive models are usually the better default. If
you only need a density/likelihood with exact inversion on low-dimensional data, a
classic normalizing flow may be simpler. See [[normalizing-flows]], [[diffusion-models]].

## 2. Mental Model

Picture every noise sample and every data sample as a dot, and imagine a **wind field**
that blows noise dots into the shape of the data. Flow Matching's trick is to define,
for each *individual* pairing of a noise point $x_0$ and a data point $x_1$, a trivially
simple path between them — a straight line — and the velocity along that line is just the
constant $x_1 - x_0$.

> **The path is your choice; the network just learns to average it.**

The magic of *conditional* flow matching: you can't see the true global wind field, but
you *can* cheaply sample one straight-line path and its constant velocity. If the network
regresses those per-sample velocities with MSE, it provably learns the correct **marginal**
velocity field — the wind that transports the whole noise cloud onto the whole data cloud —
because the MSE-optimal prediction at a point is the conditional average of the targets
passing through it.

Sampling is then "drop a noise dot, follow the wind from $t{=}0$ to $t{=}1$": integrate
the ODE with a handful of Euler/Heun steps. Straight conditional paths ⇒ nearly straight
marginal trajectories ⇒ few steps needed.

## 3. Key Concepts

- **Probability path $p_t$.** A continuous family of distributions interpolating
  $p_0$ (noise) at $t{=}0$ to $p_1$ (data) at $t{=}1$.
- **Velocity / vector field $v(x,t)$.** Generates $p_t$ via the ODE
  $\dot x = v(x,t)$ and the *continuity equation*. This is what the network predicts.
- **Conditional flow matching (CFM).** Instead of the intractable marginal field,
  regress the *conditional* field of a path tied to a single data point $x_1$. The
  gradients match those of the (intractable) marginal objective, so it's a valid,
  cheap surrogate. This is the key enabling result.
- **Conditional path & interpolant.** The common choice is the linear/Gaussian path
  $x_t = (1-t)\,x_0 + t\,x_1$ with $x_0\sim\mathcal N(0,I)$. Then the conditional
  velocity is the constant $u_t = x_1 - x_0$ — nothing to compute.
- **Rectified Flow.** Flow matching with exactly this straight-line interpolant; paths
  are straight, so few ODE steps suffice. "Reflow" can re-straighten further.
- **Training objective.** $\;\mathcal L = \mathbb E_{t,x_0,x_1}\big[\lVert v_\theta(x_t,t) - (x_1-x_0)\rVert^2\big]$ — a single MSE. Simulation-free.
- **Sampling.** Numerically integrate $\dot x = v_\theta(x,t)$ from $t{=}0$ to $1$
  (Euler, Heun, RK4, or an adaptive solver). Deterministic given $x_0$.
- **Relation to diffusion.** Diffusion's probability-flow ODE is a *special case* of an
  FM path (a particular curved noise schedule). FM lets you pick straighter paths.

## 4. Setup

Everything here is tiny and **CPU-friendly** — a small MLP velocity field on 2-D toy
data. We need only `torch`, `numpy`, and `matplotlib`. Install if you're in a fresh
environment (uncomment), otherwise the imports below are all you need.

In [1]:
# %pip install torch numpy matplotlib
import os
import math
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)

torch 2.12.1 | device: cpu


## 5. Worked Examples

### Example 1 — Train a conditional flow-matching velocity field

Target $p_1$: a ring of **8 Gaussians** (the classic 2-D toy). Base $p_0$: standard
Gaussian. We use the rectified/linear interpolant $x_t=(1-t)x_0+t x_1$, whose
conditional target velocity is the constant $x_1-x_0$, and regress it with MSE.

In [2]:
def sample_target(n):
    """8 Gaussians arranged on a ring — our data distribution p_1."""
    centers = torch.tensor(
        [[math.cos(2 * math.pi * k / 8), math.sin(2 * math.pi * k / 8)] for k in range(8)],
        dtype=torch.float32,
    ) * 4.0
    idx = torch.randint(0, 8, (n,))
    return centers[idx] + 0.25 * torch.randn(n, 2)


class VelocityField(nn.Module):
    """v_theta(x, t): R^2 x [0,1] -> R^2. Time is concatenated as an extra input."""
    def __init__(self, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 2),
        )

    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=-1))


model = VelocityField().to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)

n_steps, batch = 3000, 1024
for step in range(n_steps):
    x1 = sample_target(batch).to(device)          # data
    x0 = torch.randn_like(x1)                       # noise (base)
    t = torch.rand(batch, 1, device=device)         # time ~ U(0,1)
    xt = (1 - t) * x0 + t * x1                       # point on the linear path
    target_v = x1 - x0                               # conditional velocity (constant)
    pred_v = model(xt, t)
    loss = ((pred_v - target_v) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 500 == 0 or step == n_steps - 1:
        print(f"step {step:4d}  loss {loss.item():.4f}")

step    0  loss 9.1842


step  500  loss 3.9160


step 1000  loss 3.8034


step 1500  loss 3.5018


step 2000  loss 3.7094


step 2500  loss 3.5496


step 2999  loss 3.6001


The loss drops and plateaus quickly — note it does **not** go to zero, and shouldn't:
many different $(x_0,x_1)$ pairs pass through the same $(x_t,t)$, so the best the
network can do is predict their *average* velocity. That conditional average is exactly
the correct marginal field.

### Example 2 — Sample by integrating the ODE, and check it matches the target

Drop standard-Gaussian points at $t{=}0$ and follow $\dot x = v_\theta(x,t)$ to $t{=}1$
with simple Euler steps. We then compare each generated point's nearest ring-center
distance against true samples — and see that even **a handful of steps** works, because
the straight-line interpolant makes trajectories nearly straight.

In [3]:
@torch.no_grad()
def sample(model, n, n_steps):
    """Euler-integrate the learned ODE from noise (t=0) to data (t=1)."""
    x = torch.randn(n, 2, device=device)
    dt = 1.0 / n_steps
    for k in range(n_steps):
        t = torch.full((n, 1), k * dt, device=device)
        x = x + model(x, t) * dt
    return x


centers = torch.tensor(
    [[math.cos(2 * math.pi * k / 8), math.sin(2 * math.pi * k / 8)] for k in range(8)],
    dtype=torch.float32,
) * 4.0

def mean_dist_to_nearest_center(pts):
    d = torch.cdist(pts.cpu(), centers)          # (n, 8)
    return d.min(dim=1).values.mean().item()

real = sample_target(4000)
print(f"real data   mean dist to nearest mode: {mean_dist_to_nearest_center(real):.3f}")
for steps in (2, 5, 20, 100):
    gen = sample(model, 4000, steps)
    print(f"{steps:3d} Euler steps mean dist to nearest mode: "
          f"{mean_dist_to_nearest_center(gen):.3f}")

real data   mean dist to nearest mode: 0.313
  2 Euler steps mean dist to nearest mode: 1.265
  5 Euler steps mean dist to nearest mode: 0.499
 20 Euler steps mean dist to nearest mode: 0.364
100 Euler steps mean dist to nearest mode: 0.345


A reference real-data spread is ~0.3 (the per-mode Gaussian noise we added). Generated
samples land close to that with only a handful of steps and essentially match it by ~20 —
few-step sampling is the headline practical win of straight-path flow matching. Let's
visualize the noise→data transport to make it concrete.

In [4]:
import matplotlib
matplotlib.use("Agg")  # headless: no display needed, keeps the notebook executable anywhere
import matplotlib.pyplot as plt

gen = sample(model, 2000, 50).cpu()
real = sample_target(2000)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].scatter(real[:, 0], real[:, 1], s=3, alpha=0.4, color="tab:blue")
ax[0].set_title("Target  p_1  (8 Gaussians)")
ax[1].scatter(gen[:, 0], gen[:, 1], s=3, alpha=0.4, color="tab:green")
ax[1].set_title("Generated via ODE (50 steps)")
for a in ax:
    a.set_xlim(-6, 6); a.set_ylim(-6, 6); a.set_aspect("equal")
plt.tight_layout()
plt.savefig("flow_matching_samples.png", dpi=80)
plt.close(fig)
print("saved flow_matching_samples.png — generated cloud should reproduce all 8 modes")

saved flow_matching_samples.png — generated cloud should reproduce all 8 modes


### (Optional) Use a maintained library

For real work, don't hand-roll the solver. Meta's `flow_matching` and the
`torchcdiffeq`/`torchdyn` ecosystems give tested paths, schedulers, and adaptive ODE
solvers. The cell below is **gated** so the notebook still runs without the extra
dependency.

In [5]:
if os.getenv("RUN_FLOW_MATCHING_LIB"):
    # pip install flow_matching
    from flow_matching.path import CondOTProbPath          # optimal-transport (linear) path
    from flow_matching.solver import ODESolver
    path = CondOTProbPath()
    sample_t = path.sample(t=torch.rand(4, 1), x_0=torch.randn(4, 2), x_1=torch.randn(4, 2))
    print("path sample fields:", sample_t.x_t.shape, sample_t.dx_t.shape)
    solver = ODESolver(velocity_model=model)               # wraps our trained v_theta
    out = solver.sample(x_init=torch.randn(16, 2), method="midpoint", step_size=0.05)
    print("library-sampled shape:", out.shape)
else:
    print("Set RUN_FLOW_MATCHING_LIB=1 (and `pip install flow_matching`) to run the library path.")
    print("Our from-scratch loop above already shows the full train+sample cycle.")

Set RUN_FLOW_MATCHING_LIB=1 (and `pip install flow_matching`) to run the library path.
Our from-scratch loop above already shows the full train+sample cycle.


## 6. Gotchas & Pitfalls

- **The loss won't hit zero — that's correct.** Targets are *conditional* velocities;
  the network can only learn their average at each $(x_t,t)$. A plateauing positive loss
  is the expected outcome, not under-training.
- **Time conditioning matters.** $v_\theta$ must actually depend on $t$. Feed $t$ in
  (concatenation here; a sinusoidal/Fourier time embedding helps for harder data). Drop
  it and you collapse all paths into one and learn garbage.
- **Sample $t\sim U(0,1)$ broadly.** Under-sampling near $t{=}0$ or $t{=}1$ leaves the
  field untrained where the ODE starts/ends. Uniform is the safe default; some schedules
  reweight $t$ but don't accidentally starve the endpoints.
- **Path choice = sampling cost.** Curved paths (e.g. diffusion's variance-preserving
  schedule) need more ODE steps. Straight (rectified/OT) paths integrate in few steps —
  that's the whole point. Don't pair a curved path with a 4-step solver and blame the model.
- **Coupling matters for straightness.** Vanilla FM pairs $x_0,x_1$ *independently at
  random*, so marginal trajectories still cross/curve. Minibatch-OT coupling or
  rectified-flow "reflow" straightens them further for true 1–2 step sampling.
- **Don't confuse training with sampling.** Training is simulation-free (no ODE solve).
  Only *sampling* integrates the ODE. Putting a solver in the training loop reintroduces
  exactly the cost FM was designed to avoid.
- **Scale / normalize your data.** Like diffusion, FM assumes the base Gaussian and data
  live on comparable scales. Standardize inputs (or train in a normalized latent space).
- **Euler is fine for a demo, not always for production.** A 2nd-order (Heun/midpoint) or
  adaptive solver gives the same quality in fewer function evals on harder distributions.

## 7. When to Use vs Alternatives

| Approach | Training | Sampling | Best when |
|---|---|---|---|
| **Flow Matching / Rectified Flow** | Simulation-free MSE on velocity; very stable | Deterministic ODE, **few steps** with straight paths | Continuous data, you want fast deterministic sampling and a simple objective |
| **Diffusion (score / DDPM)** | Simulation-free denoising MSE; mature tooling | SDE or ODE, often many steps (distillation needed for few-step) | You want the most battle-tested ecosystem & conditioning tricks today |
| **Classic Normalizing Flows** | Exact max-likelihood | Exact, invertible, one pass | You need exact density/likelihood or exact inversion, lower-dim data |
| **GANs** | Adversarial (can be unstable) | **One** forward pass (fastest) | Single-step sampling is paramount and you can tolerate training difficulty / mode drop |
| **VAEs** | ELBO (fast) | One pass | Quick, smooth latent space; lower fidelity than the above |

**Honest trade-offs.** FM and diffusion are deeply related — diffusion's probability-flow
ODE is a particular FM path — so they reach similar quality; FM's edge is a *simpler
derivation* and *straighter paths* (fewer sampling steps without a separate distillation
stage). Versus GANs, FM trades single-step speed for much more stable training and full
mode coverage. Versus classic flows, FM gives up exact likelihood (you'd need an ODE
divergence integral to recover it) but scales to high dimensions far more easily.
See [[diffusion-models]], [[normalizing-flows]], [[gan]], [[vae]].

## 8. Resources

- **Lipman et al., "Flow Matching for Generative Modeling" (ICLR 2023)** — the founding
  paper (CFM objective, conditional paths): https://arxiv.org/abs/2210.02747
- **Liu et al., "Flow Straight and Fast: Rectified Flow" (ICLR 2023)** — straight-line
  paths and reflow for few-step sampling: https://arxiv.org/abs/2209.03003
- **Lipman et al., "Flow Matching Guide and Code" (2024)** — book-length tutorial + the
  official Meta `flow_matching` library: https://arxiv.org/abs/2412.06264
- **Meta `flow_matching` library (GitHub)** — tested paths, schedulers, ODE solvers:
  https://github.com/facebookresearch/flow_matching
- **Tong et al., "Conditional Flow Matching" / TorchCFM** — minibatch-OT couplings and a
  clean reference implementation: https://github.com/atong01/conditional-flow-matching
- **Esser et al., "Scaling Rectified Flow Transformers" (SD3, 2024)** — rectified flow at
  production scale: https://arxiv.org/abs/2403.03206